# Transcript Concordance by Biotype

Stacked horizontal bar plot showing transcript concordance categories
(full, partial, none) broken down by biotype. Two panels:
- **Left**: RBH-pass genes only (coverage ≥ 95%)
- **Right**: All genes with an RBH pair (regardless of coverage threshold)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

# PDF font settings — TrueType so text is editable in Illustrator / Inkscape
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# Configuration
RESULTS_DIR = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results')
LINKS_DIR = RESULTS_DIR / 'intermediate_spreadsheets' / 'sankey_plus_divergence'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colour scheme (consistent with sankey figure)
CONCORDANCE_COLORS = {
    'full':    '#1b4f72',   # Dark blue   (matches COLORS['high_confidence'])
    'partial': '#85c1e9',   # Light blue  (matches COLORS['partial'])
    'none':    '#e74c3c',   # Red         (matches COLORS['discordant'])
}
CONCORDANCE_ORDER  = ['full', 'partial', 'none']
CONCORDANCE_LABELS = {'full': 'Full match', 'partial': 'Partial', 'none': 'No match'}

BIOTYPE_ORDER  = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA':         'lncRNA',
    'pseudogene':     'Pseudogene',
    'other_ncRNA':    'Other ncRNA',
    'other':          'Other',
}

In [ ]:
# Load per-gene links
links = pd.read_csv(LINKS_DIR / 'links_per_assembly.tsv', sep='\t')
print(f'Loaded {len(links):,} rows from links_per_assembly.tsv')
print(f'Assemblies: {links["assembly_accession"].nunique()}')
print(f'Columns: {list(links.columns)}')

# ---- Version A: RBH-pass only ----
rbh_pass = links[links['rbh_status'] == 'pass'].dropna(subset=['tx_concordance']).copy()
ct_pass = (
    rbh_pass
    .groupby(['biotype', 'tx_concordance'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=BIOTYPE_ORDER, columns=CONCORDANCE_ORDER, fill_value=0)
)
print(f'\nRBH-pass cross-tab:\n{ct_pass}')

# ---- Version B: All RBH genes ----
all_with_conc = links.dropna(subset=['tx_concordance']).copy()
ct_all = (
    all_with_conc
    .groupby(['biotype', 'tx_concordance'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=BIOTYPE_ORDER, columns=CONCORDANCE_ORDER, fill_value=0)
)
print(f'\nAll-RBH cross-tab:\n{ct_all}')

In [ ]:
# ---- Stacked horizontal bar plot ----

def draw_biotype_concordance_bar(ax, ct, title):
    """Draw a stacked horizontal bar chart on *ax*."""
    y_labels = [BIOTYPE_LABELS.get(b, b) for b in BIOTYPE_ORDER]
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    for conc in CONCORDANCE_ORDER:
        vals = ct[conc].values.astype(float)
        bars = ax.barh(
            y_pos, vals, left=left,
            color=CONCORDANCE_COLORS[conc],
            label=CONCORDANCE_LABELS[conc],
            edgecolor='white', linewidth=0.5,
        )
        # Add count labels inside segments where wide enough
        for j, (v, l) in enumerate(zip(vals, left)):
            total_row = float(ct.loc[BIOTYPE_ORDER[j]].sum())
            if total_row == 0:
                continue
            pct = v / total_row * 100
            if pct >= 8:  # only label if segment is wide enough
                ax.text(
                    l + v / 2, j,
                    f'{int(v):,}\n({pct:.0f}%)',
                    ha='center', va='center',
                    fontsize=7.5, fontweight='bold', color='white',
                )
        left += vals

    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=10)
    ax.set_xlabel('Gene count (summed across assemblies)', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_yaxis()  # protein-coding at top


fig, (ax_pass, ax_all) = plt.subplots(1, 2, figsize=(18, 5.5), sharey=True)

draw_biotype_concordance_bar(ax_pass, ct_pass, 'RBH-pass genes only (coverage \u2265 95%)')
draw_biotype_concordance_bar(ax_all, ct_all, 'All genes with RBH pair')

# Remove duplicate y-labels on right panel since sharey=True
ax_all.set_yticklabels([])

# ---- Criteria text ----
criteria = (
    'Criteria \u2014 '
    'RBH pass (left panel): reciprocal locus overlap coverage \u2265 95% on both Ensembl and CAT sides. '
    'All RBH (right panel): all genes with a reciprocal best-hit pair regardless of coverage threshold. '
    'Full match: transcript concordance rate = 1.0 (all exon structures matched bidirectionally). '
    'Partial: 0 < concordance rate < 1.0 (some but not all transcripts matched). '
    'No match: concordance rate = 0 (no transcript structures matched). '
    'Biotypes grouped: protein_coding; lncRNA (incl. lnc_RNA); pseudogene (incl. pseudogenic); '
    'other_ncRNA (snRNA, snoRNA, miRNA, tRNA, rRNA, etc.); other (all remaining). '
    'Counts summed across all pangenome assemblies.'
)
fig.text(0.5, -0.04, criteria, ha='center', fontsize=8, color='#555555',
         style='italic', wrap=True)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main3_biotype_concordance.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_main3_biotype_concordance.pdf', bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_DIR / "figure_main3_biotype_concordance.png"}')
print(f'Saved to {OUTPUT_DIR / "figure_main3_biotype_concordance.pdf"}')